In [3]:
import pandas as pd
import numpy as np

In [4]:
## Phase 1 :
# Load the raw messy data
df = pd.read_csv('Raw_Nigeria_Car_Sales_Data.CSV')

In [5]:
## Phase 2 :
#Business Audit: Check for data quality issues
print("---initial data info---")
print(df.info())
print("\n---Sample of Car_Brand Mistamched---")
print(df[["Car_Brand", "Car_Model"]].head(10))

---initial data info---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8100 entries, 0 to 8099
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Sale_Date            8100 non-null   object 
 1   State                8100 non-null   object 
 2   City                 8100 non-null   object 
 3   Car_Brand            8100 non-null   object 
 4   Car_Model            8100 non-null   object 
 5   Car_Type             8100 non-null   object 
 6   Unit_Price_Naira     8100 non-null   int64  
 7   Quantity             8100 non-null   int64  
 8   Discount_Percent     8100 non-null   float64
 9   Revenue_Naira        8100 non-null   float64
 10  Customer_Type        8100 non-null   object 
 11  Delivery_Date        8100 non-null   object 
 12  Delivery_Status      8100 non-null   object 
 13  Delivery_Time_Hours  8100 non-null   int64  
 14  Warehouse            8100 non-null   object 
 15  Driver        

In [6]:
## Phase 3 :
# Define the master relationship between Brands and Models
brand_map = {
    'Toyota': ['Corolla', 'Camry', 'Highlander', 'Prado', 'Hilux', 'Venza', 'Sienna', 'Rav4'],
    'Honda': ['Accord', 'Civic', 'CR-V', 'Pilot',],
    'Benz': ['C300', 'GLE 450', 'E350', 'E-Class', 'GLC 350', 'GLK 350', 'C-Class'],
    'Lexus': ['ES 350', 'RX 350', 'GX 460','IS 250',],
    'Hyundai': ['Elantra', 'Santa Fe', 'Tucson', 'Sonata'],
    'Ford': ['Explorer', 'Edge', 'F-150', 'Focus'],
    'KIA': ['Rio', 'Sportage', 'Sorento', 'Cerato']
}

# Create a lookup dictionary
model_to_brand = {model: brand for brand, models in brand_map.items() for model in models}

# Fix: Map the correct Car_Brand to the Car_Model column
df['Brand'] = df['Car_Model'].map(model_to_brand)

print("Car_Brand/Model synchronization complete.")
# Check for NaNs again
print(f"Number of NaN values remaining: {df['Brand'].isna().sum()}")
# List every model that currently has a NaN Car_brand
missing_models = df[df['Brand'].isna()]['Car_Model'].unique()
print(missing_models)

Car_Brand/Model synchronization complete.
Number of NaN values remaining: 0
[]


In [7]:
## Phase 4 :
#
# Define estimated transit hours from Lagos (Business logic)
transit_hours_map = {
    'Lagos': 6, 'Ogun': 12, 'Oyo': 18, 'Osun': 22, 'Ekiti': 24, 'Ondo': 26,
    'Edo': 30, 'Anambra': 36, 'Enugu': 40, 'Imo': 42, 'Rivers': 45, 'Bayelsa': 48,
    'Benue': 55, 'FCT': 60, 'Kano': 72, 'Katsina': 84
}

# Apply base hours + random traffic/processing delay (2-10 hours)
np.random.seed(42)
df['Delivery_Time_Hours'] = df['State'].map(transit_hours_map) + np.random.randint(2, 10, size=len(df))

# Set Delivery Status: Business Rule: Over 50 hours is 'Delayed'
df['Delivery_Status'] = np.where(df['Delivery_Time_Hours'] > 50, 'Delayed', 'On-Time')

print("Logistics and Lead Time calculations complete.")

Logistics and Lead Time calculations complete.


In [8]:
## Phase 5 :
# Business Rule: Cap unrealistic discounts (anything over 12% is set to 4.5% average)
df['Discount_Percent'] = df['Discount_Percent'].apply(lambda x: x if x <= 12 else 4.5)

# Financial integrity: Recalculate Revenue (Price * Quantity minus Discount)
df['Revenue_Naira'] = (df['Unit_Price_Naira'] * df['Quantity']) * (1 - df['Discount_Percent']/100)

print("Financial records normalized.")

Financial records normalized.


In [9]:
## Phase 6 :
# Convert to datetime objects
df['Sale_Date'] = pd.to_datetime(df['Sale_Date'])

# Drop the old column
if "Car_Brand" in df.columns:
    df = df.drop(columns=['Car_Brand'])
# Sorting data chronologically
df = df.sort_values(by='Sale_Date').reset_index(drop=True)# Final Visual Check
print("\n--- Cleaned Data Sample ---")
# Convert all column names to lower case
df.columns = df.columns.str.lower()
print(df.head())


--- Cleaned Data Sample ---
   sale_date  state      city   car_model      car_type  unit_price_naira  \
0 2021-01-01   Ondo       Owo      Sonata  Foreign Used          16857142   
1 2021-01-01   Kano     Wudil       Camry  Foreign Used          23428571   
2 2021-01-01  Lagos  Surulere       Camry  Foreign Used          13571428   
3 2021-01-01    Imo      Orlu  Highlander  Foreign Used          16857142   
4 2021-01-02  Benue    Otukpo     Elantra  Foreign Used           6000000   

   quantity  discount_percent  revenue_naira customer_type delivery_date  \
0         1              9.27   1.529448e+07    Individual    2021-01-06   
1         1              4.50   2.237429e+07    Individual    2021-01-03   
2         1              3.68   1.307200e+07    Individual    2021-01-09   
3         1             10.74   1.504668e+07    Individual    2021-01-06   
4         1              9.21   5.447400e+06    Individual    2021-01-04   

  delivery_status  delivery_time_hours warehouse   

In [10]:
## Phase 8 :
# Export to CSV
df.to_csv('Cleaned_Nigeria_Car_Sales_Data.csv', index=False)

print("Success! File saved as 'Cleaned_Nigeria_Car_Sales_Data.csv'.")

Success! File saved as 'Cleaned_Nigeria_Car_Sales_Data.csv'.
